# 20 — Iterative RAG

> **Tier 4 | Agentic & Self-Improving**

## What is Iterative RAG?

Standard RAG retrieves once and generates. **Iterative RAG** treats the first
answer as a draft, then asks the LLM to identify what information is still
missing, generates targeted follow-up queries for each gap, retrieves fresh
evidence, and refines the answer — repeating until the answer is complete
or a convergence criterion is met.

### The loop

```
1.  Retrieve initial context (top-K)
2.  Generate draft answer
3.  Gap analysis: "What questions remain unanswered?"
4.  For each gap → new retrieval query → fetch new passages
5.  Merge new evidence with existing context (dedup)
6.  Regenerate refined answer
7.  Repeat from 3 until no gaps remain OR max iterations reached
```

This is distinct from Self RAG (nb 19) which re-evaluates *quality* of a fixed
answer. Iterative RAG re-evaluates *coverage* and adds new evidence each round.

## PDF Framework: pdfminer.six

This notebook introduces **pdfminer.six** — the most low-level of the PDF libraries.

| Feature | pdfminer.six advantage |
|---------|----------------------|
| Layout analysis | Full `LTPage` → `LTTextBox` → `LTTextLine` → `LTChar` hierarchy |
| Character-level | Font name, size, bbox per character |
| Text flow | Detects reading order via bounding-box sort |
| No C dependency | Pure Python — most portable |
| `LAParams` | Tunable line/word/char margin parameters |

We use `extract_pages()` with `LAParams` to extract `LTTextBox` objects,
preserving their layout position. Each box's `y` coordinate gives its
vertical position — we tag chunks with `y_position` (top/middle/bottom of page)
which correlates with document structure (headers at top, body in middle,
footers at bottom).

## Flow Diagram

```mermaid
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pdfminer.six"]
        PDF(["📄 climate.pdf"])
        PDF --> PM["extract_pages()\nLAParams layout analysis"]
        PM --> BOXES["LTTextBox objects\ny_position per box"]
        BOXES --> SPLIT["Chunks with\ny_position, box_count"]
        SPLIT --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\niterative_rag_20")]
    end

    subgraph ITER ["🔄  ITERATIVE RETRIEVAL LOOP"]
        Q(["❓ Query"])
        Q --> RET0["Initial retrieve\ntop-K"]
        QDRANT --> RET0
        RET0 --> DRAFT["Generate\ndraft answer"]
        DRAFT --> GAP["Gap Analyser:\nWhat is still missing?"]
        GAP --> CHK{{"Gaps\nfound?"}}
        CHK -->|"No gaps / max iter"| FINAL(["✅ Final answer"])
        CHK -->|"Yes"| SUBQ["Gap → sub-query"]
        SUBQ --> RETR["Retrieve fresh\nevidence per gap"]
        RETR --> MERGE["Merge + dedup\nall evidence"]
        MERGE --> REFINE["Regenerate\nrefined answer"]
        REFINE --> GAP
    end

    style INDEX fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style ITER  fill:#fef9c3,stroke:#ca8a04,color:#713f12
```


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pdfminer.six"]
        PDF(["📄 climate.pdf"])
        PDF --> PM["extract_pages()\nLAParams layout analysis"]
        PM --> BOXES["LTTextBox objects\ny_position per box"]
        BOXES --> SPLIT["Chunks with\ny_position, box_count"]
        SPLIT --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\niterative_rag_20")]
    end

    subgraph ITER ["🔄  ITERATIVE RETRIEVAL LOOP"]
        Q(["❓ Query"])
        Q --> RET0["Initial retrieve\ntop-K"]
        QDRANT --> RET0
        RET0 --> DRAFT["Generate\ndraft answer"]
        DRAFT --> GAP["Gap Analyser:\nWhat is still missing?"]
        GAP --> CHK{{"Gaps\nfound?"}}
        CHK -->|"No gaps / max iter"| FINAL(["✅ Final answer"])
        CHK -->|"Yes"| SUBQ["Gap → sub-query"]
        SUBQ --> RETR["Retrieve fresh\nevidence per gap"]
        RETR --> MERGE["Merge + dedup\nall evidence"]
        MERGE --> REFINE["Regenerate\nrefined answer"]
        REFINE --> GAP
    end

    style INDEX fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style ITER  fill:#fef9c3,stroke:#ca8a04,color:#713f12
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = ["boto3", "qdrant-client", "opensearch-py", "requests-aws4auth",
            "strands-agents", "pdfminer.six", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


In [ ]:
import os, json, time, uuid, re
from typing import List, Dict, Tuple, Optional
from dotenv import load_dotenv

import boto3
from pdfminer.high_level import extract_pages
from pdfminer.layout import LAParams, LTTextBox, LTTextLine, LTPage
from strands import Agent
from strands.models.bedrock import BedrockModel
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print("Imports OK")


## Step 2 — Configuration

In [ ]:
# AWS / Bedrock
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

# Vector DB
QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

# Collection
COLLECTION_NAME = "iterative_rag_20"
EMBEDDING_DIM   = 1024
TOP_K           = 5       # initial retrieval
GAP_TOP_K       = 3       # passages fetched per gap

# Chunking
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50

# Iterative RAG controls
MAX_ITERATIONS = 3    # max gap-fill cycles
MAX_GAPS       = 3    # max gaps to chase per iteration (avoid runaway)

# PDF
PDF_PATH = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection     : {COLLECTION_NAME}")
print(f"PDF            : {PDF_PATH}  (exists={os.path.exists(PDF_PATH)})")
print(f"Max iterations : {MAX_ITERATIONS}")
print(f"Max gaps/iter  : {MAX_GAPS}")


## Step 3 — Vector Store

In [ ]:
class VectorStore:
    def __init__(self, collection_name, qdrant_url='', qdrant_api_key='',
                 opensearch_url='', region='us-east-1'):
        self.name     = collection_name
        self._backend = None
        if qdrant_url:
            try:
                kw = {'url': qdrant_url}
                if qdrant_api_key: kw['api_key'] = qdrant_api_key
                self._qdrant = QdrantClient(**kw)
                self._qdrant.get_collections()
                self._backend = 'qdrant_cloud'
                print(f'Qdrant Cloud: {qdrant_url}')
                return
            except Exception as e:
                print(f'Qdrant Cloud unavailable ({e}), trying next...')
        if opensearch_url:
            try:
                from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
                creds = boto3.Session().get_credentials()
                auth  = AWSV4SignerAuth(creds, region, 'aoss')
                host  = opensearch_url.replace('https://','').replace('http://','')
                self._os = OpenSearch(
                    hosts=[{'host': host, 'port': 443}],
                    http_auth=auth, use_ssl=True, verify_certs=True,
                    connection_class=RequestsHttpConnection, timeout=30)
                self._os.info()
                self._backend = 'opensearch'
                print(f'OpenSearch: {opensearch_url}')
                return
            except Exception as e:
                print(f'OpenSearch unavailable ({e}), falling back...')
        self._qdrant  = QdrantClient(':memory:')
        self._backend = 'qdrant_memory'
        print('Using Qdrant in-memory')

    def create_collection(self, dim=1024, force_recreate=False):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            exists = self.name in [c.name for c in self._qdrant.get_collections().collections]
            if exists and force_recreate:
                self._qdrant.delete_collection(self.name); exists = False
            if not exists:
                self._qdrant.create_collection(
                    self.name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
                print(f'Created "{self.name}" (dim={dim})')
            else:
                print(f'"{self.name}" already exists')

    def upsert(self, docs: List[Dict]) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            pts = [PointStruct(id=str(uuid.uuid4()), vector=d['embedding'],
                               payload={'text': d['text'], 'metadata': d.get('metadata', {})})
                   for d in docs]
            self._qdrant.upsert(collection_name=self.name, points=pts)
            return len(pts)

    def search(self, qvec: List[float], top_k: int = 5) -> List[Dict]:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            resp = self._qdrant.query_points(
                collection_name=self.name, query=qvec, limit=top_k, with_payload=True)
            return [{'text': p.payload.get('text', ''),
                     'metadata': p.payload.get('metadata', {}),
                     'score': p.score, 'id': str(p.id)}
                    for p in resp.points]

    def count(self) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            return self._qdrant.get_collection(self.name).points_count or 0

    def delete_collection(self):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            self._qdrant.delete_collection(self.name)
        print(f'Deleted "{self.name}"')

print("VectorStore defined.")


## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

_model = BedrockModel(model_id=LLM_MODEL, region_name=AWS_REGION)

def llm(prompt: str, system: str = "You are a helpful assistant.") -> str:
    return str(Agent(model=_model, system_prompt=system)(prompt))

test_emb = embed_text("iterative rag pdfminer test")
print(f"Embedding OK — dim={len(test_emb)}")
print("Strands BedrockModel ready.")


## Step 5 — PDF Loading with pdfminer.six

`extract_pages()` yields `LTPage` objects. Each page contains `LTTextBox` elements
sorted in reading order by pdfminer's layout analysis engine.

We record the **normalised vertical position** of each box (`y_pos_norm`: 0=bottom, 1=top),
then classify it as `top` / `middle` / `bottom` zone — a structural signal
stored alongside each chunk.


In [ ]:
def recursive_split(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    separators = ["\n\n", "\n", ". ", " ", ""]
    def _split(text, seps):
        sep, new_seps = '', []
        for i, s in enumerate(seps):
            if s == '' or s in text:
                sep, new_seps = s, seps[i+1:]; break
        parts = text.split(sep) if sep != '' else list(text)
        good = []
        for part in parts:
            if len(part) <= chunk_size: good.append(part)
            elif new_seps: good.extend(_split(part, new_seps))
            else:
                for k in range(0, len(part), chunk_size - chunk_overlap):
                    good.append(part[k:k+chunk_size])
        chunks, cur_pieces, cur_len = [], [], 0
        for piece in good:
            p = piece.strip()
            if not p: continue
            addition = len(sep) + len(p) if cur_pieces else len(p)
            if cur_len + addition <= chunk_size:
                cur_pieces.append(p); cur_len += addition
            else:
                if cur_pieces: chunks.append(sep.join(cur_pieces))
                ov, ovl = [], 0
                for prev in reversed(cur_pieces):
                    if ovl + len(prev) + len(sep) <= chunk_overlap:
                        ov.insert(0, prev); ovl += len(prev) + len(sep)
                    else: break
                cur_pieces = ov + [p]
                cur_len = sum(len(x) + len(sep) for x in cur_pieces)
        if cur_pieces: chunks.append(sep.join(cur_pieces))
        return [c for c in chunks if c.strip()]
    return _split(text, separators)


def y_zone(y_norm: float) -> str:
    if y_norm >= 0.75: return 'top'
    if y_norm >= 0.35: return 'middle'
    return 'bottom'


def load_pdf_pdfminer(path: str) -> Tuple[List[Dict], Dict]:
    """
    Uses pdfminer.six extract_pages() with LAParams.
    Returns chunks with y_position zone and box_count metadata.
    """
    laparams = LAParams(
        line_margin=0.5,    # lines within this vertical distance are joined
        word_margin=0.1,    # characters within this horizontal distance form a word
        char_margin=2.0,    # characters within this distance form a line
        boxes_flow=0.5,     # weighting between horizontal/vertical for reading order
    )

    chunks     = []
    page_stats = []

    for page_num, page_layout in enumerate(extract_pages(path, laparams=laparams), start=1):
        page_h = page_layout.height or 1.0

        # Collect text boxes with their normalised y-position
        boxes = []
        for element in page_layout:
            if isinstance(element, LTTextBox):
                text = element.get_text().strip()
                if not text:
                    continue
                # y0 is bottom of bbox in pdfminer coordinates (origin = bottom-left)
                y_norm = element.y0 / page_h
                boxes.append({'text': text, 'y_norm': y_norm, 'zone': y_zone(y_norm)})

        # Sort top-to-bottom (descending y_norm = reading order)
        boxes.sort(key=lambda b: -b['y_norm'])

        # Build page text from boxes
        page_text = '\n'.join(b['text'] for b in boxes)
        if not page_text.strip():
            continue

        # Weighted y_norm for the page (avg of box y_norms)
        avg_y = sum(b['y_norm'] for b in boxes) / max(len(boxes), 1)

        page_chunks = recursive_split(page_text, CHUNK_SIZE, CHUNK_OVERLAP)
        for chunk in page_chunks:
            chunks.append({
                'text'      : chunk,
                'page_num'  : page_num,
                'y_norm'    : round(avg_y, 3),
                'y_zone'    : y_zone(avg_y),
                'box_count' : len(boxes),
                'char_count': len(chunk),
                'word_count': len(chunk.split()),
            })

        page_stats.append({
            'page': page_num, 'boxes': len(boxes),
            'avg_y': round(avg_y, 3), 'chunks': len(page_chunks)
        })

    stats = {
        'n_pages'      : len(page_stats),
        'n_chunks'     : len(chunks),
        'avg_chunk_len': sum(c['char_count'] for c in chunks) // max(len(chunks), 1),
        'zone_counts'  : {z: sum(1 for c in chunks if c['y_zone'] == z)
                          for z in ('top', 'middle', 'bottom')},
    }
    return chunks, stats, page_stats


t0 = time.time()
chunks, stats, page_stats = load_pdf_pdfminer(PDF_PATH)
elapsed = (time.time() - t0) * 1000

print(f"pdfminer.six extraction : {elapsed:.0f}ms")
print(f"Pages                   : {stats['n_pages']}")
print(f"Chunks                  : {stats['n_chunks']}")
print(f"Avg chunk length        : {stats['avg_chunk_len']} chars")
print(f"Zone distribution       : {stats['zone_counts']}")
print()
print(f"{'Page':>4}  {'Boxes':>5}  {'avg_y':>6}  {'Chunks':>6}")
print("-" * 28)
for p in page_stats:
    print(f"{p['page']:>4}  {p['boxes']:>5}  {p['avg_y']:>6.3f}  {p['chunks']:>6}")


## Step 6 — Connect & Index

In [ ]:
vs = VectorStore(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL, qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL, region=AWS_REGION)
print(f"Backend: {vs._backend}")

vs.create_collection(dim=EMBEDDING_DIM, force_recreate=True)

print(f"Embedding {len(chunks)} chunks...")
t0   = time.time()
embs = embed_batch([c['text'] for c in chunks], label='[index]')

docs = [
    {
        'text'     : chunks[i]['text'],
        'embedding': embs[i],
        'metadata' : {
            'chunk_idx' : i,
            'page_num'  : chunks[i]['page_num'],
            'y_norm'    : chunks[i]['y_norm'],
            'y_zone'    : chunks[i]['y_zone'],
            'box_count' : chunks[i]['box_count'],
            'char_count': chunks[i]['char_count'],
            'word_count': chunks[i]['word_count'],
            'source'    : 'climate.pdf',
            'pdf_lib'   : 'pdfminer.six',
        }
    }
    for i in range(len(chunks))
]
vs.upsert(docs)
print(f"Indexed {vs.count()} vectors in {time.time()-t0:.1f}s")


## Step 7 — Gap Analyser

The gap analyser takes the current question, the draft answer so far, and the
passages already used. It returns a JSON list of gap questions — specific
sub-questions whose answers are missing from the current draft.

An empty list means the answer is complete.


In [ ]:
GAP_SYSTEM = """You are an information gap analyser.
Given a question, the current draft answer, and the evidence passages used,
identify what specific information is STILL MISSING from the answer.

Return ONLY a valid JSON array of gap questions (strings).
Each gap must be a specific, answerable sub-question.
If the answer is complete, return an empty array [].
Maximum {max_gaps} gaps.

Examples:
  Input : "What causes rainfall?"  Draft: "Rainfall is caused by condensation."
  Output: ["What triggers condensation in the atmosphere?", "What role do clouds play?"]

  Input : "What is 2+2?"  Draft: "2+2 equals 4."
  Output: []
""".format(max_gaps=MAX_GAPS)


def find_gaps(question: str, draft: str, used_passages: List[str]) -> List[str]:
    evidence_summary = '\n'.join(f'- {p[:150]}...' for p in used_passages[:5])
    prompt = (
        f"Question: {question}\n\n"
        f"Current draft answer:\n{draft}\n\n"
        f"Evidence used (summaries):\n{evidence_summary}\n\n"
        f"What specific information is still missing? Return JSON array:"
    )
    raw = llm(prompt, system=GAP_SYSTEM).strip()
    m   = re.search(r'\[[\s\S]*?\]', raw)
    if m:
        try:
            gaps = json.loads(m.group())
            return [g for g in gaps if isinstance(g, str)][:MAX_GAPS]
        except json.JSONDecodeError:
            pass
    return []


# Unit test
_q      = "What is weather forecasting and how does it work?"
_draft  = "Weather forecasting is the prediction of future atmospheric conditions."
_gaps   = find_gaps(_q, _draft, ["Weather forecasting uses historical data."])
print(f"Gap test — found {len(_gaps)} gaps:")
for i, g in enumerate(_gaps, 1):
    print(f"  {i}. {g}")


## Step 8 — Generator & Evidence Merger

`generate_answer()` builds a prompt from all accumulated passages and the
original question, optionally noting the gaps being addressed.

`merge_passages()` deduplicates new passages against the existing pool using
a simple fingerprint on the first 80 characters.


In [ ]:
GEN_SYSTEM = (
    "You are a precise Q&A assistant. Answer using ONLY the provided context passages. "
    "Be comprehensive — address every part of the question. "
    "If some information is not present say so briefly, then answer what you can."
)

def generate_answer(question: str, passages: List[str], gaps_addressed: Optional[List[str]] = None) -> str:
    context = '\n\n'.join(f'[Passage {i+1}]\n{p}' for i, p in enumerate(passages))
    gap_note = ''
    if gaps_addressed:
        gap_note = '\n\nPlease ensure your answer also addresses: ' + '; '.join(gaps_addressed)
    prompt = f"Context:\n{context}\n\nQuestion: {question}{gap_note}\n\nAnswer:"
    return llm(prompt, system=GEN_SYSTEM).strip()


def merge_passages(existing: List[str], new_hits: List[Dict]) -> Tuple[List[str], int]:
    """Add new passages not already in existing pool. Returns (merged, n_added)."""
    seen      = {p[:80] for p in existing}
    merged    = list(existing)
    n_added   = 0
    for h in new_hits:
        fp = h['text'][:80]
        if fp not in seen:
            seen.add(fp)
            merged.append(h['text'])
            n_added += 1
    return merged, n_added


## Step 9 — Full Iterative RAG Pipeline

`iterative_rag()` runs the complete loop:

1. Initial retrieval → draft answer
2. Gap analysis → sub-queries
3. Per-gap retrieval → evidence merge → refined answer
4. Repeat until no gaps or `MAX_ITERATIONS` reached


In [ ]:
def iterative_rag(question: str, top_k: int = TOP_K, verbose: bool = True) -> Dict:
    t0 = time.time()

    # ── Step 1: Initial retrieval & draft ──
    init_hits = vs.search(embed_text(question), top_k=top_k)
    passages  = [h['text'] for h in init_hits]
    draft     = generate_answer(question, passages)

    if verbose:
        print(f"\nQ: {question}")
        print(f"Initial passages: {len(passages)}")
        print(f"Draft: {draft[:120]}...")

    history = [{
        'iteration': 0,
        'n_passages': len(passages),
        'gaps': [],
        'answer': draft,
    }]

    for iteration in range(1, MAX_ITERATIONS + 1):
        # ── Step 2: Find gaps ──
        gaps = find_gaps(question, draft, passages)

        if verbose:
            print(f"\n  Iter {iteration} — gaps found: {len(gaps)}")
            for i, g in enumerate(gaps, 1):
                print(f"    {i}. {g}")

        if not gaps:
            if verbose:
                print(f"  No gaps — converged at iteration {iteration}")
            break

        # ── Step 3: Retrieve evidence for each gap ──
        new_passages_added = 0
        for gap in gaps:
            gap_hits           = vs.search(embed_text(gap), top_k=GAP_TOP_K)
            passages, n_added  = merge_passages(passages, gap_hits)
            new_passages_added += n_added
            if verbose:
                print(f"    gap '{gap[:50]}...' → +{n_added} new passages")

        # ── Step 4: Regenerate with enriched evidence ──
        draft = generate_answer(question, passages, gaps_addressed=gaps)

        if verbose:
            print(f"  Total passages: {len(passages)} (+{new_passages_added} new)")
            print(f"  Refined draft: {draft[:120]}...")

        history.append({
            'iteration'  : iteration,
            'n_passages' : len(passages),
            'gaps'       : gaps,
            'n_new'      : new_passages_added,
            'answer'     : draft,
        })

        if new_passages_added == 0:
            if verbose:
                print(f"  No new evidence found — stopping")
            break

    latency = (time.time() - t0) * 1000

    if verbose:
        print(f"\nFinal answer ({latency:.0f}ms, {len(history)-1} gap-fill iterations):")
        print(draft)
        print("-" * 70)

    return {
        'question'         : question,
        'answer'           : draft,
        'iterations'       : len(history) - 1,
        'final_n_passages' : len(passages),
        'converged'        : len(history) > 1 and not history[-1].get('gaps'),
        'latency_ms'       : latency,
        'history'          : history,
    }


# Demo
result = iterative_rag("What are the main methods and tools used in weather forecasting?")


## Step 10 — Multi-Question Demo

In [ ]:
test_questions = [
    # Multi-faceted — should benefit most from gap-filling
    "Explain the complete process from data collection to issuing a weather forecast.",
    # Specific enough that initial retrieval likely suffices
    "What is nowcasting?",
    # Broad — will likely generate multiple gaps
    "How do satellites, radar, and ground stations work together in weather observation?",
]

all_results = []
for q in test_questions:
    r = iterative_rag(q, verbose=True)
    all_results.append(r)

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"{'Question':<52} {'Iters':>5}  {'Passages':>8}  {'ms':>7}")
print("-" * 76)
for r in all_results:
    print(f"{r['question'][:51]:<52} {r['iterations']:>5}  "
          f"{r['final_n_passages']:>8}  {r['latency_ms']:>7.0f}")


## Step 11 — Evidence Growth Trace

Show how the evidence pool grows across iterations for the most complex question.


In [ ]:
# Use result with most iterations for an interesting trace
trace = max(all_results, key=lambda r: r['iterations'])

print(f"Evidence growth trace for:\n  '{trace['question']}'")
print()
print(f"{'Iter':>5}  {'Passages':>8}  {'Gaps found':>10}  {'New added':>9}")
print("-" * 38)

for h in trace['history']:
    gaps_found = len(h.get('gaps', []))
    new_added  = h.get('n_new', '-')
    print(f"{h['iteration']:>5}  {h['n_passages']:>8}  {gaps_found:>10}  {str(new_added):>9}")

print()
print("Gaps identified per iteration:")
for h in trace['history']:
    if h.get('gaps'):
        print(f"  Iter {h['iteration']}:")
        for g in h['gaps']:
            print(f"    • {g}")


## Step 12 — Iterative RAG vs Simple RAG

Compare how much additional evidence iterative gap-filling adds versus a single-pass retrieval at the same `TOP_K`.


In [ ]:
def simple_rag(question: str, top_k: int = TOP_K) -> Dict:
    """Single-pass RAG baseline."""
    t0    = time.time()
    hits  = vs.search(embed_text(question), top_k=top_k)
    texts = [h['text'] for h in hits]
    ans   = generate_answer(question, texts)
    return {
        'question'  : question,
        'answer'    : ans,
        'n_passages': len(texts),
        'latency_ms': (time.time() - t0) * 1000,
    }

print("Running simple-RAG baseline for the same questions...")
simple_results = [simple_rag(q) for q in test_questions]

print()
print(f"{'Question':<52}  {'Simple P':>8}  {'Iter P':>7}  {'Gain':>5}  {'Iter ms':>8}  {'Simp ms':>8}")
print("-" * 95)
for s, r in zip(simple_results, all_results):
    gain = r['final_n_passages'] - s['n_passages']
    print(f"{r['question'][:51]:<52}  {s['n_passages']:>8}  "
          f"{r['final_n_passages']:>7}  {gain:>+5}  "
          f"{r['latency_ms']:>8.0f}  {s['latency_ms']:>8.0f}")

print()
avg_gain = sum(r['final_n_passages'] - s['n_passages']
               for r, s in zip(all_results, simple_results)) / len(all_results)
print(f"Average evidence gain from gap-filling: +{avg_gain:.1f} passages")


## Step 13 — Summary

### What Iterative RAG does differently

| Dimension | Simple RAG | Iterative RAG |
|-----------|-----------|---------------|
| Retrieval passes | 1 | 1 + up to MAX_ITERATIONS |
| Evidence breadth | Fixed TOP_K passages | Grows as gaps are closed |
| Gap handling | Ignored — answer may miss sub-topics | Explicit LLM-identified gaps drive follow-up queries |
| Convergence | N/A | Stops when no gaps remain or no new evidence found |
| Latency | Low | Higher (1–3 extra LLM + embed calls per gap) |
| Best for | Narrow, specific questions | Broad, multi-faceted questions |

### PDF Framework: pdfminer.six highlights

| Feature used | Why it matters here |
|-------------|-------------------|
| `LAParams` tuning | Controls reading-order reconstruction — critical for structured documents |
| `LTTextBox.y0` | Normalised vertical position tags chunk location (header/body/footer) |
| `box_count` per page | Density proxy for information-rich pages |
| Pure Python | No binary dependency — reproducible across OS environments |

### PDF Framework Progression

| Notebook | Pattern | Framework | Key capability leveraged |
|----------|---------|-----------|--------------------------|
| 18 | Corrective RAG | **pdfplumber** | `bbox_density` as grader signal |
| 19 | Self RAG | **pymupdf** | Speed + font-size/bold → `is_heading` |
| 20 | Iterative RAG | **pdfminer.six** | `LTTextBox.y0` → layout zone tagging |
| 21 | Recursive RAG | **unstructured** | Semantic element classification |
| 22 | Graph RAG | **docling** | Table + relationship extraction |
| 23 | Speculative RAG | **marker** | Markdown-quality layout preservation |

> **Next: [21 — Recursive RAG](../tier4_agentic/21_Recursive_RAG.ipynb)** — `unstructured` with semantic element types (Title, NarrativeText, Table) drives recursive document decomposition.


In [ ]:
print(f"Collection '{COLLECTION_NAME}': {vs.count()} vectors indexed")
print(f"PDF framework : pdfminer.six (LTTextBox layout tree, y_position zones)")
print(f"RAG pattern   : Iterative RAG (gap-analysis loop, evidence accumulation)")
print(f"Max iterations: {MAX_ITERATIONS}  |  Max gaps/iter: {MAX_GAPS}")
print()
print("Notebook 20 — Iterative RAG complete.")
